# 01 — smolagents Basics

*Study Notebook 1 of 3 · about 30 minutes*

smolagents is Hugging Face's small library for building agents. Its core idea: **let the model write
Python to act, instead of forcing it to speak in rigid JSON.**

Roadmap:
1. **Step 2** — meet the two agent types (`CodeAgent`, `ToolCallingAgent`) and write your own tool
2. **Step 3** — one real task, solved both ways, side by side
3. **Step 4** — what JSON tool-calling looks like raw, and how much smolagents saves you
4. **Step 5** — when to use which
5. **Step 6** — your turn

You'll need a free Groq API key from [console.groq.com](https://console.groq.com) (no card required).

---
## Step 0 — Install

In [ ]:
import sys
!{sys.executable} -m pip install -q "smolagents[toolkit]" litellm

---
## Step 1 — Load your API key

Checks your environment, Colab Secrets, and a local `.env` file in order; only prompts if it finds
nothing. The key is never written into the notebook.

In [ ]:
import os

def load_key(name: str):
    if os.environ.get(name):
        print(f"{name} loaded from environment"); return
    try:
        from google.colab import userdata
        os.environ[name] = userdata.get(name)
        print(f"{name} loaded from Colab Secrets"); return
    except Exception:
        pass
    try:
        from dotenv import load_dotenv
        load_dotenv()
        if os.environ.get(name):
            print(f"{name} loaded from .env file"); return
    except ImportError:
        pass
    import getpass
    os.environ[name] = getpass.getpass(f"{name}: ")

load_key("GROQ_API_KEY")

---
## Step 2 — The two agent types

smolagents gives you two kinds of agent:
- **`CodeAgent`** — the model acts by writing Python
- **`ToolCallingAgent`** — the model acts by emitting JSON tool calls

We'll try both, then learn to write our own tool.

### 2.1 — Your first agent: `CodeAgent`

Ask it something a plain LLM gets wrong — a fact that changes over time — and give it a web-search tool.

- `CodeAgent` — acts by writing Python
- `WebSearchTool` — a built-in search tool
- `LiteLLMModel` — the bridge to Groq's free Llama 3.3

In [ ]:
from smolagents import CodeAgent, WebSearchTool, LiteLLMModel

# Reused throughout the notebook
model = LiteLLMModel(model_id="groq/llama-3.3-70b-versatile")

# smolagents is model-agnostic -- swap the line above for any of these:
#   Hugging Face Inference (free, needs an HF token; may be blocked on some
#   corporate networks -- that's why we default to Groq):
#       from smolagents import InferenceClientModel
#       model = InferenceClientModel(model_id="meta-llama/Llama-3.3-70B-Instruct")
#   Local, offline, no key (via Ollama):
#       model = LiteLLMModel(model_id="ollama_chat/llama3.2")
#   Anthropic / OpenAI / others via LiteLLM:
#       model = LiteLLMModel(model_id="anthropic/claude-sonnet-4-5")

agent = CodeAgent(tools=[WebSearchTool()], model=model)
answer = agent.run("What is the latest stable version of Python, and when was it released?")
print("\nFINAL ANSWER:", answer)

In [ ]:
# What it actually wrote to get that answer
for step in agent.memory.steps:
    if getattr(step, "model_output", None):
        print("=== The agent wrote and ran this Python ===")
        print(step.model_output)
        break

It wrote **Python** — a line like `web_search("latest Python version")` — not a JSON form.

### 2.2 — The other style: `ToolCallingAgent`

Same task, same tool — but switch the agent class. Now the model emits a **JSON** tool call instead of
writing code. Notice it's a one-line change.

In [ ]:
from smolagents import ToolCallingAgent

tc_agent = ToolCallingAgent(tools=[WebSearchTool()], model=model)
answer = tc_agent.run("What is the latest stable version of Python?")
print("\nFINAL ANSWER:", answer)

**The difference in one line:**

| | `CodeAgent` | `ToolCallingAgent` |
|---|---|---|
| How it acts | Writes Python | Emits a JSON tool call |
| Good at | Logic: filtering, loops, combining results | One clean, single action per step |

Both are smolagents, both take the same tools, and switching between them is just the class name. Step 3
shows *why* the difference matters.

### 2.3 — Writing your own tool with `@tool`

A tool is just a Python function. The `@tool` decorator reads its type hints and docstring to build the
schema — no JSON, no config.

In [ ]:
from smolagents import tool

@tool
def word_count(text: str) -> int:
    """Count the number of words in a piece of text.

    Args:
        text: The text to count words in.
    """
    return len(text.split())

@tool
def char_count(text: str, include_spaces: bool = True) -> int:
    """Count characters in a piece of text.

    Args:
        text: The text.
        include_spaces: Whether to include spaces.
    """
    return len(text) if include_spaces else len(text.replace(" ", ""))

# They're ordinary functions too
print("Words:", word_count("code as actions"))

In [ ]:
# An agent uses them from plain English
text_agent = CodeAgent(tools=[word_count, char_count], model=model)
print(text_agent.run(
    "For 'Kerala is God's own country', give the word count and the char count without spaces."
))

---
## Step 3 — The same call, two ways

### 3.1 — A marketing task

The marketing workflow at **Grand Hyatt Kochi** wants to send a coupon code to **female customers**.
We'll use 5 customers here; in the real world this would be 100+.

Same two tools, solved with `ToolCallingAgent` and then `CodeAgent`. Watch the step counts.

In [ ]:
customers = [
    {"cust_id": "C1", "name": "subin", "gender": "M"},
    {"cust_id": "C2", "name": "ajmal", "gender": "M"},
    {"cust_id": "C3", "name": "sreejith", "gender": "M"},
    {"cust_id": "C4", "name": "vishnu", "gender": "M"},
    {"cust_id": "C5", "name": "aswathy achu", "gender": "F"},
]

def get_customers():
    """Return the list of customers."""
    return customers

def apply_coupon(cust_id, coupon_code):
    """Give a coupon to one customer."""
    return {"cust_id": cust_id, "coupon_code": coupon_code, "status": "applied"}

from smolagents import tool

@tool
def get_customers_tool() -> list:
    """Return the customer list. Each has cust_id, name, and gender."""
    return get_customers()

@tool
def apply_coupon_tool(cust_id: str, coupon_code: str) -> dict:
    """Give a coupon to one customer.

    Args:
        cust_id: Which customer.
        coupon_code: The coupon code.
    """
    return apply_coupon(cust_id, coupon_code)

hyatt_tools = [get_customers_tool, apply_coupon_tool]
job = "Apply coupon HYATT15 to every female customer. Report who received it."

**Way 1 — `ToolCallingAgent` (JSON)**

In [ ]:
from smolagents import ToolCallingAgent

json_agent = ToolCallingAgent(tools=hyatt_tools, model=model)
json_agent.run(job)
print(f"\nToolCallingAgent: {len(json_agent.memory.steps)} steps")

It fetches the customers, then calls `apply_coupon` once per female customer. The *"only female"*
rule can't ride inside a JSON argument, so the model checks each customer separately. Fine for 5;
costly for 100+.

**Way 2 — `CodeAgent` (Python)**

In [ ]:
from smolagents import CodeAgent

code_agent = CodeAgent(tools=hyatt_tools, model=model)
code_agent.run(job)
print(f"\nCodeAgent: {len(code_agent.memory.steps)} steps")

In [ ]:
# The Python it wrote -- filter and loop in one action
for step in code_agent.memory.steps:
    if getattr(step, "model_output", None):
        print(step.model_output)
        break

### The takeaway

| | `ToolCallingAgent` (JSON) | `CodeAgent` (code) |
|---|---|---|
| Where the "female only" rule lives | Model's reasoning, per customer | One Python line |
| Applying the coupon | One JSON call per customer | One loop |
| Trips to the model | Grows with the list | One |
| Adding a rule (e.g. "and a repeat guest") | More steps | One more `and` |

The point isn't data size — JSON can carry plenty. It's that **conditions and loops don't fit in JSON
arguments**, so they're reasoned out call by call. `CodeAgent` puts that logic in code.

---
## Step 4 — What JSON tool-calling looks like raw

`ToolCallingAgent` did the JSON work for you: it wrote a schema per tool and ran the request/response
loop. Here's the same job **without** smolagents — using the OpenAI SDK against Groq (same protocol).
This is what smolagents saves you from typing.

In [ ]:
import sys
!{sys.executable} -m pip install -q openai

In [ ]:
import json
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",  # Groq speaks the OpenAI protocol
)

# You write this schema by hand. smolagents generated it from @tool.
tools_schema = [
    {"type": "function", "function": {
        "name": "get_customers", "description": "Return the customer list.",
        "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {
        "name": "apply_coupon", "description": "Give a coupon to one customer.",
        "parameters": {"type": "object", "properties": {
            "cust_id": {"type": "string"}, "coupon_code": {"type": "string"}},
            "required": ["cust_id", "coupon_code"]}}},
]
FUNCTIONS = {"get_customers": get_customers, "apply_coupon": apply_coupon}

messages = [{"role": "user", "content": "Apply coupon HYATT15 to every female customer."}]
llm_calls = tool_calls = 0
for _ in range(12):
    llm_calls += 1
    resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile", messages=messages, tools=tools_schema)
    msg = resp.choices[0].message
    if not msg.tool_calls:
        print(f"\nDone: {llm_calls} LLM round trips, {tool_calls} tool calls.")
        break
    messages.append(msg)
    for tc in msg.tool_calls:
        tool_calls += 1
        args = json.loads(tc.function.arguments)
        out = FUNCTIONS[tc.function.name](**args)
        print(f"  round {llm_calls}: {tc.function.name}({args})")
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(out)})

That's the raw version: a hand-written schema and a loop you maintain yourself. smolagents'
`ToolCallingAgent` gave you all of this from `@tool` plus one line — and `CodeAgent` skips the
per-action round trips entirely.

---
## Step 5 — When to reach for which

- **`CodeAgent`** — tasks with logic: filtering, loops, combining results, a little math. Most real work.
  It's the default.
- **`ToolCallingAgent`** — a single, well-defined action per step, or a model/setup that doesn't handle
  code well.

Quick head-to-head on the same task:

In [ ]:
task = "Count the words in each of: 'onam', 'vishu', 'thiruvathira'. Return all three."

ca = CodeAgent(tools=[word_count], model=model)
print("CodeAgent:", ca.run(task), "|", len(ca.memory.steps), "steps")

ta = ToolCallingAgent(tools=[word_count], model=model)
print("ToolCallingAgent:", ta.run(task), "|", len(ta.memory.steps), "steps")

Same answer; `CodeAgent` usually takes fewer steps by looping over the three words in one go.

---
## Step 6 — Your turn

A conditional task: give free shipping to orders above ₹2000. Run it with **both** agents and compare
the steps. Try adding a second rule (e.g. *"and placed in Kochi"*) and watch how each agent copes.

In [ ]:
orders = [
    {"order_id": "O1", "amount": 1500, "city": "Kochi"},
    {"order_id": "O2", "amount": 3200, "city": "Kochi"},
    {"order_id": "O3", "amount": 2500, "city": "Thrissur"},
    {"order_id": "O4", "amount": 900,  "city": "Kochi"},
]

@tool
def get_orders_tool() -> list:
    """Return the orders. Each has order_id, amount (INR), and city."""
    return orders

@tool
def grant_free_shipping_tool(order_id: str) -> dict:
    """Grant free shipping to one order.

    Args:
        order_id: Which order.
    """
    return {"order_id": order_id, "free_shipping": True}

shipping_tools = [get_orders_tool, grant_free_shipping_tool]
shipping_job = "Grant free shipping to every order above 2000 rupees. List the order ids."

ca = CodeAgent(tools=shipping_tools, model=model)
ca.run(shipping_job)
print(f"CodeAgent: {len(ca.memory.steps)} steps\n")

ta = ToolCallingAgent(tools=shipping_tools, model=model)
ta.run(shipping_job)
print(f"ToolCallingAgent: {len(ta.memory.steps)} steps")

---
## Why smolagents is good for prototyping

- **Small and readable.** The core is ~1,000 lines; little hidden magic.
- **Tools are just functions.** Type hints + docstring = a tool. No schema files (you saw the raw
  alternative in Step 4).
- **Model-agnostic.** Groq now, local Ollama or Claude later — one line changes.
- **Tool interop.** Pull tools from MCP servers, LangChain, or a Hugging Face Space.
- **Transparent.** `agent.memory.steps` shows the exact code written and each tool's result — easy to debug.
- **Safer code execution.** Generated code runs through a restricted executor, not raw `exec`. For
  production, use a real sandbox (Docker, E2B).

It optimises for getting an agent working fast and understanding it — ideal for prototyping, and it
pairs with LangGraph (next notebook) when you need structure.

---
## What to remember

| Idea | In one line |
|------|-------------|
| `CodeAgent` | Model writes Python — filters, loops, combines tools in one action |
| `ToolCallingAgent` | Model writes JSON — one clean action per step |
| JSON's real limit | Conditions and loops can't live in JSON args; they're reasoned out call by call |
| `@tool` | Type hints + docstring become the schema automatically |
| `LiteLLMModel` | One line to point smolagents at Groq, Anthropic, OpenAI, and more |

**Next:** `02_langgraph_basics.ipynb` — add structure and a human approval gate around agents like these.